# Train PhoBertConceptStateTagger trên Colab

Notebook này chỉ lo phần môi trường (clone code, gắn dữ liệu/model từ Drive, cài thư viện) —
logic train thật sự nằm trong repo (`src/extraction/phobert_concept_state_tagger.py`,
`scripts/concept_state/train_phobert_concept_state_tagger.py`), đồng bộ qua git.

**Trước khi chạy:**
1. Đổi `DRIVE_ROOT` bên dưới nếu bạn để dữ liệu ở thư mục Drive khác.
2. Đã upload sẵn `data/dataset/concept_state.csv` vào `DRIVE_ROOT/data/dataset/concept_state.csv`
   trên Drive (sinh file này bằng `run.py` từ `data/ensemble/concept_state/consensus.csv`,
   không chỉnh sửa tay qua Excel/Sheets — dễ làm hỏng quoting/field CSV).
3. Chọn Runtime > Change runtime type > GPU trước khi chạy (nếu có GPU free trên Colab).


## 1. Mount Google Drive

In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 2. Clone / pull code từ git

Repo public, không cần token. Nếu đã clone từ lần trước, cell này sẽ `git pull` thay vì clone lại.

In [2]:
REPO_URL = "https://github.com/thnghia-ctu/CausalGraph.git"
BRANCH = "v3"
REPO_DIR = "/content/CausalGraph"

import os

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git checkout $BRANCH
    !git pull origin $BRANCH
else:
    !git clone -b $BRANCH $REPO_URL $REPO_DIR
    %cd $REPO_DIR


Cloning into '/content/CausalGraph'...
remote: Enumerating objects: 920, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 920 (delta 98), reused 115 (delta 42), pack-reused 718 (from 1)
Receiving objects: 100% (920/920), 1.90 MiB | 19.30 MiB/s, done.
Resolving deltas: 100% (481/481), done.
/content/CausalGraph


## 3. Gắn `data/` và `models/` vào Drive

Hai thư mục này bị `.gitignore`, không nằm trong git — clone xong sẽ trống hoặc không tồn tại.
Symlink sang Drive để dữ liệu và checkpoint được giữ lại qua các session, không cần copy tay
mỗi lần mở lại Colab.

In [3]:
DRIVE_ROOT = "/content/drive/MyDrive/CausalGraph"

import os

os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)

!rm -rf {REPO_DIR}/data {REPO_DIR}/models
!ln -s {DRIVE_ROOT}/data {REPO_DIR}/data
!ln -s {DRIVE_ROOT}/models {REPO_DIR}/models

!ls -la {REPO_DIR}/data/dataset/concept_state.csv 2>/dev/null || echo "Chưa có data/dataset/concept_state.csv trên Drive — upload trước khi train."


-rw------- 1 root root 850104 Aug 11 15:58 /content/CausalGraph/data/dataset/concept_state.csv


## 4. Cài thư viện

In [4]:
!pip install -q -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 18.1 MB/s eta 0:00:00
   ━━━

## 5. (Tuỳ chọn) Đăng nhập Hugging Face Hub

Cần nếu muốn push model lên Hub — bước 6 (train) bên dưới sẽ tự push lên Hub ngay sau khi
train xong. Token tạo tại https://huggingface.co/settings/tokens (quyền write).

In [5]:
from huggingface_hub import notebook_login

notebook_login()


## 6. Train

Checkpoint được lưu định kỳ vào `models/concept_state_tagger/` (= Drive, qua symlink ở bước 3).
Nếu Colab bị ngắt kết nối giữa chừng, chỉ cần chạy lại cell này — `PhoBertConceptStateTagger.fit`
tự resume từ checkpoint gần nhất thay vì train lại từ đầu.

In [6]:
!python -m scripts.concept_state.train_phobert_concept_state_tagger


config.json: 100% 557/557 [00:00<00:00, 2.21MB/s]
vocab.txt: 100% 895k/895k [00:00<00:00, 12.0MB/s]
bpe.codes: 100% 1.14M/1.14M [00:00<00:00, 14.2MB/s]
tokenizer.json: 100% 3.13M/3.13M [00:00<00:00, 21.4MB/s]

pytorch_model.bin: downloading bytes:  17% 90.5M/543M [00:01<00:02, 153MB/s, 5.77MB/s  ]
pytorch_model.bin: downloading bytes:  44% 236M/543M [00:02<00:02, 145MB/s, 20.2MB/s  ]
pytorch_model.bin: downloading bytes:  60% 324M/543M [00:02<00:00, 225MB/s, 26.0MB/s  ]
pytorch_model.bin: downloading bytes:  64% 350M/543M [00:02<00:01, 187MB/s, 30.3MB/s  ]
pytorch_model.bin: downloading bytes: 100% 366M/366M [00:03<00:00, 106MB/s, 30.8MB/s  ]
pytorch_model.bin: reconstructing file: 100% 543M/543M [00:03<00:00, 157MB/s, 47.5MB/s  ]
Loading weights: 100% 197/197 [00:00<00:00, 17308.60it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | U